# PLN Healthtec — execução reproduzível
**Autores: Raphael Di Giorgio e Aquiles Esaú da Silva Reis**

## Como executar
1. Abra este notebook no Colab e use **Ambiente de execução → Executar tudo** para uma nova execução completa. A instalação pode exigir reiniciar o runtime; se solicitado, reinicie e execute novamente.
2. Espere o **Bloco 16** terminar; baixe o ZIP pelo botão da última célula. Ele contém as tabelas, matrizes, erros, modelos, versões, dados, embeddings e divisões.
3. Use `resumo_resultados.md` e `comparacao_modelos.csv` desse pacote para atualizar o PDF. **Os números históricos do PDF não são resultados garantidos desta versão.** As saídas antigas foram limpas para evitar mistura de execuções.
4. Para repetir somente a comparação, em ambiente com a mesma versão de Python e dependências do `requirements-lock.txt` do ZIP, carregue o ZIP na aba Arquivos do Colab. Execute apenas a célula de definições do Bloco 16, informe o caminho em `REPLAY_ZIP` na célula seguinte e execute-a. Não é necessário refazer SBERT, LDA ou Louvain.

### Protocolo e limites
- Cinco configurações: LinearSVC/SBERT, Regressão Logística/SBERT, k-NN/SBERT, Naive Bayes/TF-IDF e LinearSVC/TF-IDF, para ambos os alvos.
- Split estratificado 80/20; CV-5 estratificada somente no treino; sementes 42. TF-IDF é aprendido dentro de cada fold e, depois, somente no treino completo.
- Para corresponder à metodologia descrita, LDA/Louvain e undersampling continuam anteriores ao split. **Isso é uma avaliação transdutiva de rótulos sintéticos, não validação clínica nem avaliação de generalização de toda a pipeline.**
- O grafo mantém por padrão pesos de **distância**, conforme a implementação original e a limitação declarada no relatório. `PESO_GRAFO='similaridade'` é uma alternativa metodológica que muda a partição e exige nova interpretação; não misture suas métricas com as antigas.
- Semente de Louvain agora fixa, mas IDs e quantidades de comunidades podem diferir das execuções históricas sem semente. A execução exporta a partição efetivamente utilizada.
- O notebook compara classificadores clássicos. A assistência de IA no desenvolvimento ou na redação é uma questão distinta e deve ser declarada de acordo com as regras da disciplina.

# Bloco 1: ambiente e aquisição da amostra


In [ ]:
%pip install -q datasets pandas numpy scipy scikit-learn matplotlib seaborn spacy sentence-transformers networkx python-louvain umap-learn ipywidgets deep-translator joblib threadpoolctl
import os, random, json, platform
import numpy as np
import pandas as pd
from datasets import load_dataset
from huggingface_hub import HfApi
SEED = 42
random.seed(SEED); np.random.seed(SEED)
# Revisões imutáveis são resolvidas e registradas nesta execução.
# Para reconstrução desde os dados brutos, preencha com os hashes do manifest.json anterior.
DATASET_ID = "ruslanmv/ai-medical-dataset"
DATASET_REVISION = None
SBERT_ID = "sentence-transformers/all-MiniLM-L6-v2"
SBERT_REVISION = None
PESO_GRAFO = "distancia"  # protocolo histórico; alternativa explícita: "similaridade"
api = HfApi()
DATASET_REVISION = DATASET_REVISION or api.dataset_info(DATASET_ID).sha
SBERT_REVISION = SBERT_REVISION or api.model_info(SBERT_ID).sha
PROVENIENCIA = {'dataset':DATASET_ID,'dataset_revision':DATASET_REVISION,
               'sbert':SBERT_ID,'sbert_revision':SBERT_REVISION,
               'sampling':'primeiros 10000 exemplos do split train, sem embaralhar',
               'seed':SEED,'graph_weights':PESO_GRAFO,'louvain_seed':SEED}
# Streaming evita baixar o corpus inteiro de milhões de registros.
stream = load_dataset(DATASET_ID, split="train", revision=DATASET_REVISION, streaming=True)
df_fonte = pd.DataFrame(list(stream.take(10000)))
assert len(df_fonte)==10000, "A fonte não forneceu os 10000 registros esperados."
df_amostra = df_fonte.head(10).copy()
display(df_amostra.head())
print(json.dumps(PROVENIENCIA, indent=2, ensure_ascii=False))


In [ ]:
# As dependências são instaladas juntas no Bloco 1.
# A versão efetivamente usada será exportada pelo Bloco 16.
print("Semente global:", SEED)


#Bloco 2 Estrutura do dataset

In [ ]:
print("Dimensão da amostra carregada:", df_fonte.shape)
print("Colunas e tipos:")
print(df_fonte.dtypes)
print("Exemplo:")
display(df_amostra.iloc[0].to_dict())


# BLOCO 3: Separação da Amostra e Preparação

In [ ]:
df = df_fonte.copy()
# Mantemos o protocolo original, mas registramos um diagnóstico explícito de qualidade.
DIAGNOSTICO_DADOS = {
    'n_linhas':len(df),
    'nulos':df[['question','context']].isna().sum().to_dict(),
    'duplicatas_question_context':int(df.duplicated(['question','context']).sum()),
    'vazios':{c:int(df[c].fillna('').astype(str).str.strip().eq('').sum()) for c in ['question','context']}
}
print(DIAGNOSTICO_DADOS)
if df[['question','context']].isna().any().any():
    raise ValueError("Há nulos na amostra. Defina e documente seu tratamento antes de continuar.")
df['texto_bruto'] = df['question'].astype(str) + " " + df['context'].astype(str)
PROVENIENCIA['diagnostico_dados'] = DIAGNOSTICO_DADOS
print("Amostra:",len(df),"registros; duplicatas e vazios são reportados, sem remoção automática.")
display(df[['texto_bruto']].head())


#Bloco 4: pré processamento de dados

In [ ]:
# Instalação e download do modelo do spaCy (silencioso)
!python -m spacy download en_core_web_sm -q

import spacy
import re

# Carrega o modelo em inglês.
# Desabilitamos o 'parser' e o 'ner' (Named Entity Recognition) porque só precisamos
# da lematização e stopwords. Isso deixa o processamento até 5x mais rápido.
nlp = spacy.load("en_core_web_sm", disable=['ner', 'parser'])

def limpar_texto_medico(texto):
    """
    Função dedicada para limpeza de textos com foco em modelagem de tópicos e embeddings.
    """
    # 1. Converte para minúsculas
    texto = str(texto).lower()

    # 2. Remove números e pontuações, mantendo apenas letras e espaços
    texto = re.sub(r'[^a-z\s]', ' ', texto)

    # 3. Processamento no spaCy
    doc = nlp(texto)

    # 4. Regras de filtragem:
    # - Não pode ser stopword (is_stop)
    # - Não pode ser espaço em branco ou apenas 1 letra (len > 1)
    # - Extrai o lemma (raiz da palavra) em vez da palavra original
    tokens = [token.lemma_ for token in doc if not token.is_stop and len(token.text) > 1]

    return " ".join(tokens)

print("Iniciando o pré-processamento de 10.000 linhas...")

# List comprehension é significativamente mais rápido que o df.apply() no Pandas
df['texto_limpo'] = [limpar_texto_medico(texto) for texto in df['texto_bruto']]

print("Pré-processamento concluído!")
display(df[['texto_bruto', 'texto_limpo']].head())


# BLOCO 5: Criação do TARGET via LDA e Gráfico de Balanceamento

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Vetorização: Transforma o texto limpo em uma matriz de frequência de palavras
vectorizer = CountVectorizer(max_df=0.95, min_df=2, max_features=1000)
matriz_termos = vectorizer.fit_transform(df['texto_limpo'])

# 2. Modelagem de Tópicos (LDA): Vamos criar 5 categorias (tópicos) como exemplo
n_topicos = 5
lda_model = LatentDirichletAllocation(n_components=n_topicos, random_state=42, learning_method="batch", max_iter=10, n_jobs=1)
lda_output = lda_model.fit_transform(matriz_termos)

# 3. Criando a coluna TARGET: Pegamos o tópico com maior probabilidade para cada texto
df['TARGET'] = lda_output.argmax(axis=1)
df['TARGET'] = df['TARGET'].apply(lambda x: f"Tópico {x}")

print("Categorias (TARGET) criadas com sucesso pelo LDA!\n")
plt.figure(figsize=(10, 6))

# Usamos o df criado nas etapas anteriores
ax = sns.countplot(
    x='TARGET',
    data=df,
    hue='TARGET',
    palette="Set2",
    legend=False,
    order=sorted(df['TARGET'].unique()) # Ordena para ficar bonito no gráfico
)

# Quantidade sobre cada barra
for container in ax.containers:
    ax.bar_label(container, fontsize=12, fontweight="bold")

plt.xlabel("Classe Dominante (Tópico LDA)")
plt.ylabel("Quantidade de Textos")
plt.title("Distribuição das Classes no Dataset Médico")
sns.despine() # Deixa o gráfico mais clean
plt.show()


#Bloco 6: Interpretação dos Tópicos do LDA

In [ ]:
import numpy as np

def interpretar_topicos_lda(modelo, vetorizador, n_palavras=10):
    """
    Extrai as palavras com maior peso (probabilidade) em cada tópico gerado pelo LDA.
    """
    # Pega o vocabulário (todas as palavras únicas que o vetorizador aprendeu)
    palavras_vocabulario = np.array(vetorizador.get_feature_names_out())

    print("Principais palavras-chave por Tópico:\n")

    for idx, distribuicao_pesos in enumerate(modelo.components_):
        # Ordena os índices das palavras pelo peso (do maior para o menor)
        indices_top_palavras = distribuicao_pesos.argsort()[:-n_palavras - 1:-1]

        # Resgata as palavras reais usando os índices
        top_palavras = palavras_vocabulario[indices_top_palavras]

        # Conta quantos textos caíram nesse tópico
        qtd_textos = df[df['TARGET'] == f'Tópico {idx}'].shape[0]

        print(f"Tópico {idx} ({qtd_textos} textos):")
        print(" | ".join(top_palavras))
        print("-" * 60)

# Executando a função para ver as 10 palavras mais importantes de cada classe
interpretar_topicos_lda(lda_model, vectorizer, n_palavras=10)

#Bloco 7: Geração de Embeddings com SBERT

In [ ]:
# Instalação silenciosa da biblioteca


from sentence_transformers import SentenceTransformer
import numpy as np

# Carregando o modelo.
# O 'all-MiniLM-L6-v2' é extremamente rápido e excelente para validar a pipeline no Colab.
# Dica: Na execução final, você pode trocar por um modelo específico para o domínio médico,
# como o 'pritamdeka/S-PubMedBert-MS-MARCO'.
import torch
torch.manual_seed(SEED)
torch.set_num_threads(1)
torch.use_deterministic_algorithms(True)
modelo_sbert = SentenceTransformer(SBERT_ID, revision=SBERT_REVISION, device='cpu')
PROVENIENCIA['sbert_max_seq_length'] = modelo_sbert.max_seq_length
PROVENIENCIA['embedding_device'] = 'cpu' 

print("Iniciando a geração dos vetores densos (embeddings) para 10.000 textos...")

# O processamento em batch é feito automaticamente pelo encode()
# O parâmetro show_progress_bar ajuda a monitorar o tempo de execução
embeddings = modelo_sbert.encode(df['texto_bruto'].tolist(), show_progress_bar=True, batch_size=32, normalize_embeddings=False)

# Salvamos os embeddings como uma lista de arrays no DataFrame para facilitar cruzamentos futuros
df['embeddings'] = list(embeddings)

print(f"\nMatriz de embeddings gerada com sucesso!")
print(f"Formato da matriz (Textos, Dimensões): {embeddings.shape}")


# Bloco 8: Topologia de Redes e Detecção de Comunidades (Louvain)

In [ ]:
# Instala a biblioteca do Louvain silenciosamente


import networkx as nx
import community.community_louvain as louvain
from sklearn.neighbors import kneighbors_graph
import numpy as np

print("Construindo o grafo de similaridade (k-NN)...")
# Usa as embeddings do SBERT (Bloco 7) geradas previamente
# O k=10 significa que conectaremos cada texto apenas aos seus 10 vizinhos mais similares.
# Isso cria uma matriz esparsa, ideal para grafos grandes.
X_embeddings = np.stack(df['embeddings'].values)
grafo_knn = kneighbors_graph(X_embeddings, n_neighbors=10, mode='distance', metric='cosine', include_self=False)

if PESO_GRAFO == 'similaridade':
    grafo_knn.data = np.clip(1.0 - grafo_knn.data, 0.0, 1.0)
    grafo_knn.eliminate_zeros()
elif PESO_GRAFO != 'distancia':
    raise ValueError('PESO_GRAFO deve ser distancia ou similaridade')
print('Pesos do grafo:', PESO_GRAFO)
# Conversão mantém o procedimento original; o manifesto identifica essa escolha.
G = nx.from_scipy_sparse_array(grafo_knn)

print("Aplicando o algoritmo de Louvain para detecção de comunidades...")
# O Louvain vai otimizar a modularidade da rede e agrupar os textos em "bolhas" semânticas
particao = louvain.best_partition(G, weight='weight', resolution=1.0, random_state=SEED)

# Salva a comunidade detectada de volta no DataFrame
df['Comunidade_Louvain'] = df.index.map(particao)
n_comunidades = len(set(particao.values()))

print(f"Topologia estruturada com sucesso!")
print(f"O algoritmo de Louvain detectou {n_comunidades} comunidades distintas na rede.")


# Bloco 9: Redução de Dimensionalidade (UMAP) e Visualização Espacial

In [ ]:
# Instalação silenciosa do UMAP
!pip install -q umap-learn

import umap
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("Reduzindo vetores de 384 dimensões para 2D usando UMAP...")

# Configurando o UMAP com métrica de cosseno para alinhar com a lógica espacial do SBERT e k-NN
redutor_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=42)
embeddings_2d = redutor_umap.fit_transform(np.stack(df['embeddings'].values))

# Adicionando as novas coordenadas espaciais ao nosso DataFrame
df['umap_x'] = embeddings_2d[:, 0]
df['umap_y'] = embeddings_2d[:, 1]

print("Gerando a visualização topológica da rede...")

plt.figure(figsize=(14, 10))

# Plotagem da dispersão, utilizando uma paleta de 20 cores para as comunidades de Louvain
ax = sns.scatterplot(
    x='umap_x',
    y='umap_y',
    hue='Comunidade_Louvain',
    palette='tab20', # Perfeito para até 20 classes categóricas
    data=df,
    s=25,          # Tamanho dos pontos
    alpha=0.8,     # Transparência para ver sobreposições
    edgecolor=None
)

plt.title("Projeção Topológica UMAP: 20 Comunidades Semânticas (SBERT + Louvain)", fontsize=16, fontweight='bold')
plt.xlabel("UMAP Dimensão 1", fontsize=12)
plt.ylabel("UMAP Dimensão 2", fontsize=12)

# Ajuste da legenda para ficar fora do gráfico e não cobrir os clusters
plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left', borderaxespad=0., title="Comunidade (Louvain)")
sns.despine()
plt.tight_layout()
plt.show()

# Bloco 10: Interseção Semântica (LDA vs Louvain) e Extração de Termos

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

print("1. Cruzando os Tópicos LDA com as Comunidades Louvain...")
# Cria uma tabela cruzada para ver quantos textos de cada Tópico LDA caíram em cada Comunidade Louvain
tabela_cruzada = pd.crosstab(df['TARGET'], df['Comunidade_Louvain'])

plt.figure(figsize=(16, 6))
sns.heatmap(tabela_cruzada, annot=True, fmt="d", cmap="YlGnBu", cbar=False, linewidths=.5)
plt.title("Distribuição das Comunidades Louvain dentro dos Tópicos LDA", fontsize=14, fontweight='bold')
plt.xlabel("ID da Comunidade (Louvain)")
plt.ylabel("Macro-Área (LDA TARGET)")
plt.show()

print("\n2. Extraindo as palavras-chave das 20 Comunidades Louvain via TF-IDF...")
# O TF-IDF penaliza palavras comuns e destaca termos exclusivos de cada comunidade
tfidf_vectorizer = TfidfVectorizer(max_df=0.8, min_df=5, stop_words='english')
matriz_tfidf = tfidf_vectorizer.fit_transform(df['texto_limpo'])
palavras_tfidf = tfidf_vectorizer.get_feature_names_out()

# Cria um dicionário para armazenar o perfil de cada comunidade
perfil_comunidades = {}

for comunidade in sorted(df['Comunidade_Louvain'].unique()):
    # Isola as linhas (índices) da matriz TF-IDF que pertencem a esta comunidade
    indices_comunidade = df.index[df['Comunidade_Louvain'] == comunidade].tolist()
    tfidf_submatriz = matriz_tfidf[indices_comunidade]

    # Calcula a média do TF-IDF para cada palavra dentro desta comunidade
    pesos_medios = tfidf_submatriz.mean(axis=0).A1

    # Pega os índices das 5 palavras com maior peso
    top_indices = pesos_medios.argsort()[-5:][::-1]
    top_palavras = [palavras_tfidf[i] for i in top_indices]

    perfil_comunidades[comunidade] = top_palavras
    print(f"Comunidade {comunidade:02d} ({len(indices_comunidade):4d} textos): {', '.join(top_palavras)}")

# Bloco 11: Validação Algorítmica e Mitigação de Viés (Undersampling)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

print("Iniciando Validação Algorítmica e Mitigação de Viés...\n")

# ---------------------------------------------------------
# 1. Undersampling para as Macro-Áreas (LDA TARGET)
# ---------------------------------------------------------
# Encontra o tamanho da menor classe do LDA (Tópico 1 = 1112)
min_lda_count = df['TARGET'].value_counts().min()

# Aplica o undersampling garantindo que todas as classes tenham exatamente esse tamanho
df_balanceado_lda = df.groupby('TARGET').sample(n=min_lda_count, random_state=42)
print(f"Base LDA Balanceada: {len(df_balanceado_lda)} textos ({min_lda_count} por classe).")

# ---------------------------------------------------------
# 2. Undersampling para as Micro-Comunidades Clínicas (Louvain)
# ---------------------------------------------------------
# Filtra comunidades com menos de 100 textos; os IDs dependem desta execução.
# para evitar o colapso do tamanho do dataset
contagem_louvain = df['Comunidade_Louvain'].value_counts()
comunidades_representativas = contagem_louvain[contagem_louvain >= 100].index
df_louvain_filtrado = df[df['Comunidade_Louvain'].isin(comunidades_representativas)]

if df_louvain_filtrado.empty:
    raise ValueError('Nenhuma comunidade com pelo menos 100 exemplos.')
# Encontra a menor classe efetivamente presente nesta execução.
min_louvain_count = df_louvain_filtrado['Comunidade_Louvain'].value_counts().min()

# Aplica o undersampling nas comunidades válidas
df_balanceado_louvain = df_louvain_filtrado.groupby('Comunidade_Louvain').sample(n=min_louvain_count, random_state=42)
print(f"Base Louvain Balanceada: {len(df_balanceado_louvain)} textos ({min_louvain_count} por classe representativa).")

# ---------------------------------------------------------
# 3. Visualização Comparativa da Validação
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Plot LDA
sns.countplot(x='TARGET', data=df_balanceado_lda, ax=axes[0], palette="Set2", order=sorted(df_balanceado_lda['TARGET'].unique()))
axes[0].set_title("Classes LDA (Pós-Mitigação de Viés)", fontweight='bold')
axes[0].set_ylabel("Quantidade")
axes[0].tick_params(axis='x', rotation=45)

# Plot Louvain
sns.countplot(x='Comunidade_Louvain', data=df_balanceado_louvain, ax=axes[1], palette="tab20")
axes[1].set_title("Comunidades Louvain (Pós-Mitigação de Viés)", fontweight='bold')
axes[1].set_ylabel("")

sns.despine()
plt.tight_layout()
plt.show()


Bloco 12: Classificação SVM (Macro LDA vs Micro Louvain)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import numpy as np

print("Iniciando treinamento com SVM (Support Vector Machine) em 384 dimensões...\n")

# ---------------------------------------------------------
# 1. Preparação e Treinamento: MACRO-ÁREAS (LDA)
# ---------------------------------------------------------
X_lda = np.stack(df_balanceado_lda['embeddings'].values)
y_lda = df_balanceado_lda['TARGET'].values

X_train_lda, X_test_lda, y_train_lda, y_test_lda = train_test_split(
    X_lda, y_lda, test_size=0.20, random_state=42, stratify=y_lda
)

# max_iter=2000 garante que o modelo tenha tempo de convergir sem gerar avisos
svm_lda = LinearSVC(C=1.0, random_state=42, max_iter=2000, tol=1e-4, dual="auto")
svm_lda.fit(X_train_lda, y_train_lda)
y_pred_lda = svm_lda.predict(X_test_lda)

# ---------------------------------------------------------
# 2. Preparação e Treinamento: MICRO-COMUNIDADES (Louvain)
# ---------------------------------------------------------
X_louvain = np.stack(df_balanceado_louvain['embeddings'].values)
y_louvain = df_balanceado_louvain['Comunidade_Louvain'].values

X_train_louv, X_test_louv, y_train_louv, y_test_louv = train_test_split(
    X_louvain, y_louvain, test_size=0.20, random_state=42, stratify=y_louvain
)

svm_louvain = LinearSVC(C=1.0, random_state=42, max_iter=2000, tol=1e-4, dual="auto")
svm_louvain.fit(X_train_louv, y_train_louv)
y_pred_louv = svm_louvain.predict(X_test_louv)

# ---------------------------------------------------------
# 3. Relatórios de Desempenho
# ---------------------------------------------------------
print("=== Desempenho MACRO-ÁREAS (LDA) ===")
print(classification_report(y_test_lda, y_pred_lda))

print("\n=== Desempenho MICRO-COMUNIDADES (Louvain) ===")
print(classification_report(y_test_louv, y_pred_louv))

# ---------------------------------------------------------
# 4. Plotagem das Matrizes de Confusão Lado a Lado
# ---------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# Matriz LDA
cm_lda = confusion_matrix(y_test_lda, y_pred_lda, labels=svm_lda.classes_)
disp_lda = ConfusionMatrixDisplay(confusion_matrix=cm_lda, display_labels=svm_lda.classes_)
disp_lda.plot(cmap='Blues', values_format='d', ax=axes[0], colorbar=False)
axes[0].set_title("Matriz de Confusão: SVM (LDA - Macro)", fontweight='bold', fontsize=14)
axes[0].tick_params(axis='x', rotation=45)
axes[0].set_xlabel("Previsão")
axes[0].set_ylabel("Realidade")

# Matriz Louvain
cm_louv = confusion_matrix(y_test_louv, y_pred_louv, labels=svm_louvain.classes_)
disp_louv = ConfusionMatrixDisplay(confusion_matrix=cm_louv, display_labels=svm_louvain.classes_)
disp_louv.plot(cmap='Greens', values_format='d', ax=axes[1], colorbar=False)
axes[1].set_title("Matriz de Confusão: SVM (Louvain - Micro)", fontweight='bold', fontsize=14)
axes[1].tick_params(axis='x', rotation=0) # Rotação 0 pois os labels são números curtos
axes[1].set_xlabel("Previsão")
axes[1].set_ylabel("")

plt.tight_layout()
plt.show()


#Bloco 13: Pipeline de Inferência

In [ ]:
def classificar_texto_medico(texto_novo):
    """
    Recebe um texto médico bruto, processa e prevê sua comunidade clínica usando o SVM.
    """
    print(f"Texto Original: '{texto_novo}'\n")

    # 1. Limpeza do texto (reutilizamos a função do Bloco 4, se necessário para o modelo final)
    # Como o SBERT usa o texto_bruto, podemos passar direto, mas o pré-processamento ajuda a padronizar.

    # 2. Geração do Embedding (384D) com o SBERT já carregado
    print("Gerando representação semântica...")
    vetor_novo = modelo_sbert.encode([texto_novo])

    # 3. Predição com o SVM do Louvain (Micro-comunidades)
    pred_louvain = svm_louvain.predict(vetor_novo)[0]

    # 4. Predição com o SVM do LDA (Macro-áreas)
    pred_lda = svm_lda.predict(vetor_novo)[0]

    # Resgatando as palavras-chave da comunidade (geradas no Bloco 10) para contexto
    palavras_chave = perfil_comunidades.get(pred_louvain, ["N/A"])

    print("-" * 50)
    print("🔬 CLASSIFICAÇÃO TEMÁTICA (NÃO É DIAGNÓSTICO):")
    print(f"Macro-Área (LDA): {pred_lda}")
    print(f"Micro-Comunidade (Louvain): {pred_louvain:02d}")
    print(f"Termos Associados à Comunidade {pred_louvain:02d}: {', '.join(palavras_chave)}")
    print("-" * 50)

# ==========================================
# TESTE DE MESA (Coloque resumos reais de artigos aqui)
# ==========================================

texto_teste_1 = "The patient presented with severe chest pain, elevated troponin levels, and was immediately taken for a coronary angiography which confirmed an acute myocardial infarction."
classificar_texto_medico(texto_teste_1)

texto_teste_2 = "We analyzed the expression of BRCA1 and BRCA2 genes in human epithelial cells to understand their role in tumor suppression and rapid cell division."
classificar_texto_medico(texto_teste_2)


# Bloco 14: Exportação dos Modelos (Deploy)

In [ ]:
import joblib

print("Salvando os modelos treinados e estruturas de dados...")

# Salvando os modelos SVM
joblib.dump(svm_lda, 'modelo_svm_lda.pkl')
joblib.dump(svm_louvain, 'modelo_svm_louvain.pkl')

# Salvando o perfil de comunidades (o dicionário com as palavras-chave)
joblib.dump(perfil_comunidades, 'perfil_comunidades_louvain.pkl')

print("Exportação concluída! Verifique a aba de arquivos (ícone de pasta à esquerda no Colab) para fazer o download dos arquivos .pkl.")

# Bloco 15: Interface Interativa com Suporte Multilíngue (PT, ES, EN)

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output
from deep_translator import GoogleTranslator

# 1. Elementos Visuais
caixa_texto = widgets.Textarea(
    value='',
    placeholder='Cole aqui o relato clínico, abstract ou anotação...',
    description='Texto:',
    layout=widgets.Layout(width='90%', height='120px')
)

# NOVO: Checkbox para controlar o uso da API e evitar bloqueios
checkbox_traducao = widgets.Checkbox(
    value=False,
    description='Traduzir para Inglês (Marque apenas se o texto for em PT/ES)',
    indent=False,
    layout=widgets.Layout(width='90%', margin='0px 0px 10px 0px')
)

botao_analisar = widgets.Button(
    description='Analisar Texto',
    button_style='success',
    icon='check'
)

saida = widgets.Output()

def ao_clicar_no_botao(b):
    with saida:
        clear_output()
        texto_original = caixa_texto.value

        if not texto_original.strip():
            print("⚠️ Por favor, insira um texto para análise.")
            return

        print("⏳ Processando semântica...")

        # 2. Lógica de Roteamento (Bypass da API)
        if checkbox_traducao.value:
            try:
                tradutor = GoogleTranslator(source='auto', target='en')
                texto_modelo = tradutor.translate(texto_original)
            except Exception:
                clear_output()
                print("=" * 70)
                print("❌ O Google bloqueou temporariamente o IP do Colab para traduções.")
                print("👉 Por favor, desmarque a caixinha e cole o texto diretamente em INGLÊS.")
                print("=" * 70)
                return
        else:
            # Se não estiver marcado, o texto vai direto para a IA (Zero risco de bloqueio)
            texto_modelo = texto_original

        # 3. Passa pelo SBERT e pelo SVM
        vetor_novo = modelo_sbert.encode([texto_modelo])
        pred_louvain = svm_louvain.predict(vetor_novo)[0]
        pred_lda = svm_lda.predict(vetor_novo)[0]
        palavras_chave = perfil_comunidades.get(pred_louvain, ["N/A"])

        # 4. Impressão do Resultado
        clear_output()
        print("=" * 70)
        print("📋 TEXTO DE ENTRADA:")
        print(f"Original: {texto_original[:150]}...")
        if checkbox_traducao.value and texto_original != texto_modelo:
            print(f"Tradução: {texto_modelo[:150]}...")

        print("\n🔬 CLASSIFICAÇÃO TEMÁTICA (NÃO É DIAGNÓSTICO):")
        print(f"➤ Macro-Área (LDA): {pred_lda}")
        print(f"➤ Micro-Comunidade (Louvain): {pred_louvain:02d}")
        print(f"➤ Foco Semântico: {', '.join(palavras_chave)}")
        print("=" * 70)

botao_analisar.on_click(ao_clicar_no_botao)

print("✨ Interface Preditiva Pronta!")
display(caixa_texto, checkbox_traducao, botao_analisar, saida)


# Bloco 16: comparação de modelos, CV-5 e reprodução
## 16A — definições
Execute esta célula também quando quiser repetir apenas a comparação a partir de um ZIP.
O TF-IDF fica dentro de `Pipeline`, evitando aprender o vocabulário dos folds de validação ou do teste.
Os cinco modelos recebem o mesmo split e os mesmos folds em cada tarefa. O teste é usado somente após CV e ajuste do modelo com hiperparâmetros predefinidos.
A comparação no teste é descritiva; não selecionamos hiperparâmetros com esse conjunto.

O pacote inclui `manifest.json`, hashes SHA-256, `requirements-lock.txt`, `pip-freeze.txt`, bases, embeddings, IDs do split/folds, métricas, predições, erros, matrizes e modelos. Guarde o ZIP: sementes isoladas não substituem a preservação dos dados e do ambiente.


In [ ]:
"""Comparação reproduzível de cinco classificadores; sem resultados pré-fixados."""
from pathlib import Path
import io, json, hashlib, platform, sys, subprocess, zipfile, warnings, time, shutil
from datetime import datetime, timezone
from importlib.metadata import version, PackageNotFoundError
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
from sklearn.base import clone
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix, ConfusionMatrixDisplay
from threadpoolctl import threadpool_limits

SEED_COMPARACAO = 42
PACOTES = ['numpy','pandas','scipy','scikit-learn','joblib','threadpoolctl',
           'matplotlib','seaborn','datasets','huggingface-hub','spacy','en-core-web-sm',
           'sentence-transformers','transformers','torch','networkx','python-louvain','umap-learn']

def versoes():
    out = {}
    for p in PACOTES:
        try: out[p] = version(p)
        except PackageNotFoundError: pass
    return out

def modelos():
    # Parâmetros explícitos: mudanças de defaults da biblioteca não alteram o protocolo.
    svm = lambda: LinearSVC(C=1.0, max_iter=2000, tol=1e-4, dual='auto', random_state=42)
    tfidf = lambda: TfidfVectorizer(max_df=0.8, min_df=5)
    return {
        'svm_sbert': ('SBERT', svm()),
        'logreg_sbert': ('SBERT', LogisticRegression(C=1.0, solver='lbfgs', max_iter=2000, tol=1e-4, random_state=42)),
        'knn_sbert': ('SBERT', KNeighborsClassifier(n_neighbors=10, metric='cosine', algorithm='brute', weights='uniform', n_jobs=1)),
        'nb_tfidf': ('TF-IDF', Pipeline([('tfidf', tfidf()), ('clf', MultinomialNB(alpha=1.0))])),
        'svm_tfidf': ('TF-IDF', Pipeline([('tfidf', tfidf()), ('clf', svm())])),
    }

def preparar_base(base):
    required = {'texto_bruto','texto_limpo','embeddings'}
    if not required.issubset(base.columns):
        raise ValueError(f'Colunas ausentes: {required-set(base.columns)}. Execute os blocos anteriores.')
    if not base.index.is_unique: raise ValueError('Índices originais precisam ser únicos.')
    if base[list(required-{'embeddings'})].isna().any().any(): raise ValueError('Há textos nulos.')
    texts = base.drop(columns=['embeddings']).copy()
    texts.insert(0, 'row_id', base.index.astype(int))
    texts = texts.reset_index(drop=True)
    vectors = np.stack(base['embeddings'].to_numpy())
    if not np.isfinite(vectors).all(): raise ValueError('Embeddings contêm valores não finitos.')
    return texts, vectors

def divisao(texts, target):
    y = texts[target].to_numpy()
    tr, te = train_test_split(np.arange(len(y)), test_size=0.2, random_state=42, stratify=y)
    if pd.Series(y[tr]).value_counts().min() < 5:
        raise ValueError('Cada classe precisa de pelo menos 5 exemplos no treino para CV-5.')
    folds = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=42).split(tr, y[tr]))
    fold_id = np.full(len(y), -1, dtype=int)
    for i, (_, val) in enumerate(folds): fold_id[tr[val]] = i
    split = pd.DataFrame({'row_id': texts.row_id, 'subset': 'test', 'cv_fold': fold_id})
    split.loc[tr, 'subset'] = 'train'
    return split

def ler_checkpoint(arquivo):
    # Lê apenas CSV/NPY/JSON do pacote; não desserializa os modelos pickle.
    with zipfile.ZipFile(arquivo) as z:
        m = json.loads(z.read('manifest.json'))
        if m.get('protocol_version') != 'healthtec-comparison-v1':
            raise ValueError('Versão de protocolo incompatível.')
        for name, expected in m['sha256'].items():
            if hashlib.sha256(z.read(name)).hexdigest()!=expected:
                raise ValueError(f'Checksum divergente: {name}')
        current = versoes()
        diff = {k:(v,current.get(k)) for k,v in m['versions'].items() if current.get(k)!=v}
        if diff: raise RuntimeError(f'Ambiente diferente do checkpoint: {diff}. Use requirements-lock.txt e a versão Python registrada em ambiente separado.')
        if platform.python_version()!=m['python']:
            raise RuntimeError('Versão de Python diferente do checkpoint.')
        bases = {}
        for task, target in [('lda','TARGET'),('louvain','Comunidade_Louvain')]:
            table = pd.read_csv(io.BytesIO(z.read(f'{task}/dados.csv')), keep_default_na=False)
            vectors = np.load(io.BytesIO(z.read(f'{task}/embeddings.npy')), allow_pickle=False)
            split = pd.read_csv(io.BytesIO(z.read(f'{task}/split.csv')))
            bases[task] = (table,vectors,split,target)
    return bases, m['provenance']

def executar_comparacao(base_lda=None, base_louvain=None, replay_zip=None,
                        out_dir=None, provenance=None, mostrar=True):
    if replay_zip:
        bases, provenance = ler_checkpoint(replay_zip)
    else:
        if base_lda is None or base_louvain is None: raise ValueError('Execute os blocos 1-15 ou forneça REPLAY_ZIP.')
        bases = {}
        for task, base, target in [('lda',base_lda,'TARGET'),('louvain',base_louvain,'Comunidade_Louvain')]:
            table, vectors = preparar_base(base)
            bases[task] = (table,vectors,divisao(table,target),target)
    if out_dir is None:
        out_dir = Path('resultados_healthtec') / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=False)
    all_metrics, alerts, task_info = [], [], {}
    for task, (table, vectors, split, target) in bases.items():
        folder=out/task;folder.mkdir()
        if len(vectors)!=len(table) or not np.array_equal(table.row_id,split.row_id):
            raise ValueError('Checkpoint desalinhado: linhas, vetores e divisão.')
        if set(split.subset)!={'train','test'} or set(split.loc[split.subset=='train','cv_fold'])!=set(range(5)):
            raise ValueError('Divisão ou folds inválidos.')
        tr=np.flatnonzero(split.subset.eq('train'));te=np.flatnonzero(split.subset.eq('test'))
        y=table[target].to_numpy(); labels=np.unique(y)
        ytrain,ytest=y[tr],y[te]
        folds=[(np.flatnonzero(split.cv_fold.to_numpy()[tr]!=k),np.flatnonzero(split.cv_fold.to_numpy()[tr]==k)) for k in range(5)]
        assert set(table.row_id.iloc[tr]).isdisjoint(table.row_id.iloc[te])
        table.to_csv(folder/'dados.csv',index=False)
        np.save(folder/'embeddings.npy',vectors,allow_pickle=False)
        split.to_csv(folder/'split.csv',index=False)
        task_info[task]={'target':target,'classes':[str(v) for v in labels], 'n_classes':len(labels),
                         'n_total':len(y),'n_train':len(tr),'n_test':len(te),
                         'class_counts':{str(k):int(v) for k,v in pd.Series(y).value_counts().items()}}
        print(f'\n{task.upper()}: {len(labels)} classes | treino={len(tr)} | teste={len(te)}',flush=True)
        for name,(representation, estimator) in modelos().items():
            X=vectors if representation=='SBERT' else table.texto_limpo.to_numpy()
            Xtrain,Xtest=X[tr],X[te]
            with threadpool_limits(limits=1), warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                # Mesmo treino e mesmos folds para os cinco modelos; TF-IDF é ajustado dentro de cada fold.
                scores=cross_val_score(clone(estimator),Xtrain,ytrain,cv=folds,scoring='f1_macro',n_jobs=1,error_score='raise')
                model=clone(estimator)
                started=time.perf_counter();model.fit(Xtrain,ytrain);fit_time=time.perf_counter()-started
                started=time.perf_counter();pred=model.predict(Xtest);predict_time=time.perf_counter()-started
            for warning in caught:
                alerts.append({'task':task,'model':name,'category':warning.category.__name__,'message':str(warning.message)})
            precision,recall,f1,_=precision_recall_fscore_support(ytest,pred,average='macro',zero_division=0)
            row={'task':task,'model':name,'representation':representation,'n_classes':len(labels),'n_test':len(te),
                 'accuracy':accuracy_score(ytest,pred),'precision_macro':precision,'recall_macro':recall,'f1_macro':f1,
                 'f1_cv_mean':scores.mean(),'f1_cv_std':scores.std(ddof=1),'fit_seconds':fit_time,'predict_seconds':predict_time}
            all_metrics.append(row)
            print(name, 'F1 teste:',round(f1,4),'CV:',round(scores.mean(),4),flush=True)
            pd.DataFrame({'fold':range(5),'f1_macro':scores}).to_csv(folder/f'{name}_cv.csv',index=False)
            report=classification_report(ytest,pred,output_dict=True,zero_division=0)
            pd.DataFrame(report).T.to_csv(folder/f'{name}_por_classe.csv')
            cm=confusion_matrix(ytest,pred,labels=labels)
            pd.DataFrame(cm,index=labels,columns=labels).to_csv(folder/f'{name}_confusao.csv')
            fig,ax=plt.subplots(figsize=(max(7,len(labels)*.55),max(6,len(labels)*.5)))
            ConfusionMatrixDisplay(cm,display_labels=labels).plot(ax=ax,cmap='Blues',colorbar=False,values_format='d')
            ax.set_title(f'{task} | {name}');ax.tick_params(axis='x',rotation=90);fig.tight_layout()
            fig.savefig(folder/f'{name}_confusao.png',dpi=160);plt.close(fig)
            predictions=table.iloc[te][['row_id','texto_bruto','texto_limpo']].copy()
            predictions['real']=ytest;predictions['previsto']=pred;predictions['erro']=ytest!=pred
            predictions['n_palavras']=predictions.texto_bruto.str.split().str.len()
            # Quartis com limites por comprimento; empates podem reduzir o número de grupos.
            predictions['quartil']=pd.qcut(predictions.n_palavras,4,duplicates='drop')
            predictions.to_csv(folder/f'{name}_predicoes.csv',index=False)
            predictions[predictions.erro].to_csv(folder/f'{name}_erros.csv',index=False)
            predictions.groupby('quartil',observed=True).agg(n=('erro','size'),taxa_erro=('erro','mean'),mediana_palavras=('n_palavras','median')).to_csv(folder/f'{name}_erros_comprimento.csv')
            predictions[predictions.erro].groupby(['real','previsto']).size().sort_values(ascending=False).rename('n_erros').to_csv(folder/f'{name}_pares_confundidos.csv')
            joblib.dump(model,folder/f'{name}.pkl')
    # Modelos e perfil pertencem à mesma partição desta execução.
    shutil.copyfile(out/'lda/svm_sbert.pkl',out/'modelo_svm_lda.pkl')
    shutil.copyfile(out/'louvain/svm_sbert.pkl',out/'modelo_svm_louvain.pkl')
    if provenance and 'semantic_profile' in provenance:
        perfil={int(k):v for k,v in provenance['semantic_profile'].items()}
        joblib.dump(perfil,out/'perfil_comunidades_louvain.pkl')
    metrics=pd.DataFrame(all_metrics)
    metrics.to_csv(out/'comparacao_modelos.csv',index=False)
    (out/'avisos.json').write_text(json.dumps(alerts,ensure_ascii=False,indent=2),encoding='utf-8')
    fig,axes=plt.subplots(1,2,figsize=(16,6))
    for task,ax in zip(['lda','louvain'],axes):
        part=metrics[metrics.task.eq(task)].sort_values('f1_macro')
        ypos=np.arange(len(part))
        ax.barh(ypos,part.f1_macro,color=['#3274a1' if r=='SBERT' else '#e1812c' for r in part.representation],label='F1 teste')
        ax.errorbar(part.f1_cv_mean,ypos,xerr=part.f1_cv_std,fmt='D',color='black',label='CV-5: média ± DP')
        ax.set_yticks(ypos,part.model);ax.set_xlim(0,1);ax.set_title(task.upper());ax.legend();ax.set_xlabel('F1 macro')
    fig.tight_layout();fig.savefig(out/'comparacao_f1.png',dpi=180)
    if mostrar:plt.show()
    plt.close(fig)
    packages=versoes()
    (out/'requirements-lock.txt').write_text('\n'.join(f'{k}=={v}' for k,v in sorted(packages.items()))+'\n')
    freeze=subprocess.run([sys.executable,'-m','pip','freeze'],capture_output=True,text=True,check=True).stdout
    (out/'pip-freeze.txt').write_text(freeze)
    configuration={name:{'representation':rep,'parameters':{k:repr(v) for k,v in model.get_params(deep=True).items()}} for name,(rep,model) in modelos().items()}
    manifest={'protocol_version':'healthtec-comparison-v1','schema_version':1,'created_utc':datetime.now(timezone.utc).isoformat(),'python':platform.python_version(),
              'platform':platform.platform(),'versions':packages,'seed':42,'split':'stratified 80/20',
              'cv':'StratifiedKFold(5, shuffle=True, random_state=42), treino apenas; DP amostral ddof=1',
              'tasks':task_info,'models':configuration,'provenance':provenance or {'status':'runtime anterior; revisões não registradas'},
              'limitations':['Rótulos LDA/Louvain derivados do corpus antes do split: avaliação transdutiva.',
                             'CV avalia classificadores condicionados aos rótulos fixos, não toda a geração dos rótulos.',
                             'Balanceamento e corte de comunidades >=100 precedem o split, conforme protocolo original.',
                             'Sem validação clínica; tempos dependem de hardware; não há garantia de igualdade entre versões.'],
              'sha256':{}}
    lines=['# Resultados desta execução','', 'Autores: Raphael Di Giorgio e Aquiles Esaú da Silva Reis','',
           'Valores calculados automaticamente. Esta execução substitui as tabelas numéricas antigas do relatório; não é denominada execução A ou B.','']
    for task in ['lda','louvain']:
        p=metrics[metrics.task.eq(task)].sort_values('f1_macro',ascending=False)
        info=task_info[task]
        lines += [f'## {task.upper()}',f"Classes: {info['n_classes']}; treino: {info['n_train']}; teste: {info['n_test']}.",'',
                  '| Modelo | Acurácia | Precisão macro | Recall macro | F1 macro | CV-5 média ± DP | Treino (s) |',
                  '|---|---:|---:|---:|---:|---:|---:|']
        for _,r in p.iterrows():
            lines.append(f"| {r.model} | {r.accuracy:.4f} | {r.precision_macro:.4f} | {r.recall_macro:.4f} | {r.f1_macro:.4f} | {r.f1_cv_mean:.4f} ± {r.f1_cv_std:.4f} | {r.fit_seconds:.3f} |")
        lines += ['',f'Maior F1 observado no teste: {p.iloc[0].model}. Diferenças descritivas; sem teste de significância.','']
    lines += ['## Interpretação e limites',*['- '+t for t in manifest['limitations']],
              '- AUC não foi incluída: o protocolo prioriza métricas multiclasse macro e matrizes de confusão. Escores de decisão podem ser usados em certas formulações de AUC; probabilidades não são uma exigência universal.',
              '- Custos acima não incluem aquisição, limpeza, embeddings, LDA, grafo ou validação cruzada.',
              '- Consulte avisos.json: avisos de convergência não devem ser ignorados.',
              '- Números são resultados computacionais de rótulos sintéticos, não desempenho clínico.']
    (out/'resumo_resultados.md').write_text('\n'.join(lines),encoding='utf-8')
    for p in sorted(out.rglob('*')):
        if p.is_file():manifest['sha256'][p.relative_to(out).as_posix()]=hashlib.sha256(p.read_bytes()).hexdigest()
    (out/'manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
    archive=out.with_suffix('.zip')
    with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
        for p in sorted(out.rglob('*')):
            if p.is_file():z.write(p,p.relative_to(out))
    if mostrar:
        from IPython.display import display,FileLink
        display(metrics.round(4));display(FileLink(str(archive)));display(FileLink(str(out/'resumo_resultados.md')))
    print('Pacote de reprodução:',archive)
    return metrics,out,archive


## 16B — executar
- **Nova execução:** mantenha `REPLAY_ZIP = None` após executar os blocos anteriores.
- **Reprodução:** informe o caminho do ZIP carregado no Colab, por exemplo `/content/20261009T....zip`. Execute 16A e 16B no ambiente registrado, sem executar 1–15.
- O código interrompe a reprodução se hashes, versões ou divisão forem incompatíveis. Os tempos de execução podem variar mesmo com métricas reproduzidas.


In [ ]:
REPLAY_ZIP = None  # ou "/content/NOME_DO_PACOTE.zip"
if REPLAY_ZIP:
    comparacao, pasta_resultados, pacote_resultados = executar_comparacao(replay_zip=REPLAY_ZIP)
else:
    if not {'df_balanceado_lda','df_balanceado_louvain'}.issubset(globals()):
        raise RuntimeError('Execute os blocos anteriores ou informe REPLAY_ZIP.')
    PROVENIENCIA['semantic_profile'] = {str(k):list(v) for k,v in perfil_comunidades.items()}
    PROVENIENCIA['corpus_louvain_counts'] = {str(k):int(v) for k,v in df.Comunidade_Louvain.value_counts().items()}
    comparacao, pasta_resultados, pacote_resultados = executar_comparacao(
        df_balanceado_lda, df_balanceado_louvain, provenance=PROVENIENCIA)


## 16C — baixar e atualizar o relatório
Baixe o ZIP ao finalizar. Atualize as tabelas e a análise do PDF com esta execução, incluindo a contagem real de comunidades, o tamanho do teste e os avisos de convergência, se houver. Não reutilize as tabelas históricas A/B sem os respectivos registros de execução.

O ZIP contém os modelos recém-treinados nos subdiretórios `lda/` e `louvain/`. Os três `.pkl` históricos do GitHub não são substituídos automaticamente: não misture modelos de uma partição com perfis de outra. O Bloco 14 exporta os modelos e o perfil da nova execução completa.


In [ ]:
# Download manual, para não iniciar downloads durante "Executar tudo".
import ipywidgets as widgets
from IPython.display import display
botao_download = widgets.Button(description='Baixar resultados ZIP', button_style='success')
def baixar_resultados(_):
    try:
        from google.colab import files
        files.download(str(pacote_resultados))
    except ImportError:
        print('Abra o ZIP no caminho:', pacote_resultados)
botao_download.on_click(baixar_resultados)
display(botao_download)
